In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
import shutil
import torch.nn.functional as F
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
import scipy.io
from copy import deepcopy
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
import math
import pickle

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics import recall_score, precision_score, cohen_kappa_score

import json
import hashlib
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
import shutil
import torch.nn.functional as F
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
import scipy.io
from copy import deepcopy
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
import math
import pickle

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics import recall_score, precision_score, cohen_kappa_score

import json
import hashlib
from typing import Tuple

import numpy as np
from scipy.signal import welch

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

/home/usuario-utp/anaconda3/envs/yessica/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================================================
# 1. Positional Encoding
# =========================================================

class PositionalEncoding(nn.Module):
    """
    Fixed 1-D sinusoidal positional encoding.
    This mimics your Keras implementation.
    """

    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        self.d_model = d_model

        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)  # (L, 1)
        i = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)    # (1, D)

        exponent = (2.0 * torch.floor(i / 2.0)) / float(d_model)
        angle_rates = 1.0 / torch.pow(torch.tensor(10000.0), exponent)
        angle_rads = pos * angle_rates                                # (L, D)

        # Same behavior as your Keras code: concat sin and cos
        sin = torch.sin(angle_rads[:, 0::2])
        cos = torch.cos(angle_rads[:, 1::2])

        pe = torch.cat([sin, cos], dim=-1)  # (max_len, d_model)
        pe = pe.unsqueeze(0)                # (1, max_len, d_model)

        self.register_buffer("pos_encoding", pe)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        return x.float() + self.pos_encoding[:, :seq_len, :].to(x.device)


# =========================================================
# 2. Keras-like Multi-Head Self-Attention
# =========================================================

class KerasStyleMultiHeadSelfAttention(nn.Module):
    """
    This is closer to tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model
    )

    In your Keras code, key_dim=d_model.
    PyTorch's nn.MultiheadAttention behaves differently because it splits
    d_model across heads. This custom version is closer to Keras.
    """

    def __init__(self, d_model: int = 64, num_heads: int = 4, key_dim: int = None):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.key_dim = key_dim if key_dim is not None else d_model
        self.inner_dim = self.num_heads * self.key_dim

        self.q_proj = nn.Linear(d_model, self.inner_dim)
        self.k_proj = nn.Linear(d_model, self.inner_dim)
        self.v_proj = nn.Linear(d_model, self.inner_dim)

        self.out_proj = nn.Linear(self.inner_dim, d_model)

        self.scale = self.key_dim ** -0.5

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """

        B, L, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, L, self.num_heads, self.key_dim).transpose(1, 2)
        k = k.view(B, L, self.num_heads, self.key_dim).transpose(1, 2)
        v = v.view(B, L, self.num_heads, self.key_dim).transpose(1, 2)

        # Attention scores: (B, heads, L, L)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn, v)  # (B, heads, L, key_dim)

        context = context.transpose(1, 2).contiguous()
        context = context.view(B, L, self.inner_dim)

        output = self.out_proj(context)  # (B, L, d_model)

        return output


# =========================================================
# 3. Transformer Encoder Block
# =========================================================

class TransformerEncoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int = 64,
        num_heads: int = 4,
        d_ff: int = 128,
        rate: float = 0.1,
    ):
        super().__init__()

        self.attn = KerasStyleMultiHeadSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            key_dim=d_model,
        )

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)

    def forward(self, x):
        attn_output = self.attn(x)
        attn_output = self.dropout1(attn_output)

        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)

        out2 = self.layernorm2(out1 + ffn_output)

        return out2


# =========================================================
# 4. One Stream Encoder
# =========================================================

class StreamEncoder(nn.Module):
    """
    Equivalent to your build_stream function.
    """

    def __init__(
        self,
        seq_len: int,
        feat_dim: int,
        d_model: int = 64,
        num_layers: int = 2,
        num_heads: int = 4,
        d_ff: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.feat_dim = feat_dim

        self.projection = nn.Linear(feat_dim, d_model)
        self.pos_encoding = PositionalEncoding(d_model=d_model)

        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                rate=dropout,
            )
            for _ in range(num_layers)
        ])

        self.decoder = nn.Linear(d_model, 128)

    def forward(self, x):
        """
        x: (batch, seq_len, feat_dim)
        """

        x = self.projection(x)
        x = self.pos_encoding(x)

        for block in self.transformer_blocks:
            x = block(x)

        # Equivalent to GlobalAveragePooling1D
        x = x.mean(dim=1)

        # Equivalent to Dense(128, activation="relu")
        x = F.relu(self.decoder(x))

        return x


# =========================================================
# 5. Full Multi-Stream Model
# =========================================================

class EEGAttentionTransformer(nn.Module):
    """
    PyTorch version of EEG_Attention_Transformer.

    Inputs:
        freq: (batch, 20, 1)
        temp: (batch, 10, 1)
        spat: (batch, C, 1)

    Output:
        logits: (batch, 2)

    Important:
        For training with nn.CrossEntropyLoss, use logits directly.
        For probabilities, use torch.softmax(logits, dim=1).
    """

    def __init__(
        self,
        freq_shape: Tuple[int, int],
        temp_shape: Tuple[int, int],
        spat_shape: Tuple[int, int],
        d_model: int = 64,
        num_layers: int = 2,
        num_heads: int = 4,
        d_ff: int = 128,
        num_classes: int = 2,
    ):
        super().__init__()

        self.freq_stream = StreamEncoder(
            seq_len=freq_shape[0],
            feat_dim=freq_shape[1],
            d_model=d_model,
            num_layers=num_layers,
            num_heads=num_heads,
            d_ff=d_ff,
        )

        self.temp_stream = StreamEncoder(
            seq_len=temp_shape[0],
            feat_dim=temp_shape[1],
            d_model=d_model,
            num_layers=num_layers,
            num_heads=num_heads,
            d_ff=d_ff,
        )

        self.spat_stream = StreamEncoder(
            seq_len=spat_shape[0],
            feat_dim=spat_shape[1],
            d_model=d_model,
            num_layers=num_layers,
            num_heads=num_heads,
            d_ff=d_ff,
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, freq, temp, spat, return_proba: bool = False):
        freq_vec = self.freq_stream(freq)
        temp_vec = self.temp_stream(temp)
        spat_vec = self.spat_stream(spat)

        x = torch.cat([freq_vec, temp_vec, spat_vec], dim=1)

        logits = self.classifier(x)

        if return_proba:
            return torch.softmax(logits, dim=1)

        return logits

In [3]:
# =========================================================
# 6. Stream preparation function
# =========================================================

def prepare_streams_4s(X, fs=128, n_win=10):
    """
    X : ndarray (N, C, 512)
        4-second EEG segments sampled at 128 Hz.

    Returns:
        freq: (N, 20, 1)
        temp: (N, 10, 1)
        spat: (N, C, 1)
    """

    N, C, T = X.shape
    assert T == 512, "Expecting exactly 512-sample segments, i.e., 4 s at 128 Hz"

    # -----------------------------------------------------
    # 1. Spectral stream: 20 log-spaced bands
    # -----------------------------------------------------
    log_bands = np.logspace(np.log10(1), np.log10(fs / 2), 21)
    bands = list(zip(log_bands[:-1], log_bands[1:]))

    def band_power(signal):
        f, Pxx = welch(signal, fs=fs, nperseg=512)

        return np.array([
            Pxx[(f >= lo) & (f < hi)].mean()
            if np.any((f >= lo) & (f < hi))
            else 0.0
            for lo, hi in bands
        ])

    freq_feats = np.stack([
        [band_power(ch) for ch in trial]
        for trial in X
    ])

    freq = freq_feats.mean(axis=1)[..., None]  # (N, 20, 1)

    # -----------------------------------------------------
    # 2. Temporal stream: 10 windows
    # -----------------------------------------------------
    T_win = T // n_win        # 512 // 10 = 51
    usable = n_win * T_win   # 510

    x_trim = X[:, :, :usable]
    windows = x_trim.reshape(N, C, n_win, T_win)

    temp = windows.mean(axis=(1, 3))[..., None]  # (N, 10, 1)

    # -----------------------------------------------------
    # 3. Spatial stream: per-channel RMS
    # -----------------------------------------------------
    spat = np.sqrt((X ** 2).mean(axis=2))[..., None]  # (N, C, 1)

    return (
        freq.astype("float32"),
        temp.astype("float32"),
        spat.astype("float32"),
    )

In [4]:
# =========================================================
# 7. PyTorch Dataset
# =========================================================

class MultiStreamEEGDataset(Dataset):
    def __init__(self, freq, temp, spat, y):
        self.freq = torch.tensor(freq, dtype=torch.float32)
        self.temp = torch.tensor(temp, dtype=torch.float32)
        self.spat = torch.tensor(spat, dtype=torch.float32)

        # y debe ser entero: 0 o 1
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.freq[idx],
            self.temp[idx],
            self.spat[idx],
            self.y[idx],
        )

In [5]:
# =========================================================
# IMPORTS
# =========================================================

import torch
import torch.nn as nn
from torchinfo import summary


# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# WRAPPER PARA SUMMARY
# =========================================================

class SummaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, freq, temp, spat):
        out = self.model(freq, temp, spat)
        return out  # logits: (B, nb_classes)


# =========================================================
# CREAR MODELO MULTI-STREAM TRANSFORMER
# =========================================================

model = EEGAttentionTransformer(
    freq_shape=(20, 1),   # spectral stream: (20 bands, 1 feature)
    temp_shape=(10, 1),   # temporal stream: (10 windows, 1 feature)
    spat_shape=(19, 1),   # spatial stream: (19 channels, 1 feature)
    d_model=64,
    num_layers=2,
    num_heads=4,
    d_ff=128,
    num_classes=2,
).to(device)

wrapped_model = SummaryWrapper(model).to(device)


# =========================================================
# SUMMARY TIPO TENSORFLOW
# =========================================================

summary(
    wrapped_model,
    input_size=[
        (1, 20, 1),   # freq input: (batch, seq_len, feat_dim)
        (1, 10, 1),   # temp input
        (1, 19, 1),   # spat input
    ],
    depth=5,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    device=device,
)

Using device: cuda


Layer (type:depth-idx)                                            Input Shape               Output Shape              Param #                   Trainable
SummaryWrapper                                                    [1, 20, 1]                [1, 2]                    --                        True
├─EEGAttentionTransformer: 1-1                                    [1, 20, 1]                [1, 2]                    --                        True
│    └─StreamEncoder: 2-1                                         [1, 20, 1]                [1, 128]                  --                        True
│    │    └─Linear: 3-1                                           [1, 20, 1]                [1, 20, 64]               128                       True
│    │    └─PositionalEncoding: 3-2                               [1, 20, 64]               [1, 20, 64]               --                        --
│    │    └─ModuleList: 3-3                                       --                        --         

In [6]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).
        labels (array): Etiquetas por sujeto en formato 0/1.

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas 0/1 por segmento
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos, dtype=np.float32), np.array(y, dtype=np.int64), sbjs, window_ids


def get_segmented_data():
    """
    Carga la base de datos y devuelve los segmentos listos para T-GARNet.
    """
    ruta_carpeta_TDAH = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group'
    ruta_carpeta_control = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group'

    sujetos_TDAH = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_TDAH)
        if archivo.endswith(".mat")
    ])

    sujetos_control = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_control)
        if archivo.endswith(".mat")
    ])

    # =========================================================
    # ELIMINAR EXPLÍCITAMENTE EL SUJETO QUE NO ESTÁ EN folds.pkl
    # =========================================================
    sujeto_a_eliminar = "v36p"

    if sujeto_a_eliminar in sujetos_TDAH:
        sujetos_TDAH.remove(sujeto_a_eliminar)
        print(f"Sujeto eliminado explícitamente: {sujeto_a_eliminar}")
    else:
        print(f"ADVERTENCIA: {sujeto_a_eliminar} no está en sujetos_TDAH")

    print("Sujetos TDAH encontrados:", len(sujetos_TDAH))
    print("Sujetos Control encontrados:", len(sujetos_control))
    print("Sujetos totales encontrados:", len(sujetos_TDAH) + len(sujetos_control))

    diagnostico = {}

    for sbj in sujetos_TDAH:
        diagnostico[sbj] = 1

    for sbj in sujetos_control:
        diagnostico[sbj] = 0

    eeg_tdah = {}
    for sbj in sujetos_TDAH:
        mat_file_path = ruta_carpeta_TDAH + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T

    eeg_control = {}
    for sbj in sujetos_control:
        mat_file_path = ruta_carpeta_control + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T

    db = eeg_control | eeg_tdah

    zeros = np.zeros(len(eeg_control))
    ones = np.ones(len(eeg_tdah))
    labels = np.hstack((zeros, ones))

    X, y, sbjs, window_ids = segmentar_senales(db, labels)

    y = np.eye(2, dtype=np.float32)[y]

    return X.astype(np.float32), y.astype(np.float32), sbjs, window_ids

In [7]:
# =========================================================
# SEED
# =========================================================

def set_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

In [8]:
# =========================================================
# WRAPPER PARA COMPATIBILIDAD CON RUTINAS TIPO TGARNET / SHALLOW
# =========================================================

class MultiStreamReturnDictWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, freq, temp, spat):
        logits = self.model(freq, temp, spat)

        out_activation = torch.softmax(logits, dim=1)

        return {
            "logits": logits,
            "out_activation": out_activation,
        }

In [9]:
# =========================================================
# CONSTRUCTOR DEL MODELO
# =========================================================

def build_eeg_model(model_name, **kwargs):
    model_name = model_name.lower()

    valid_names = [
        "multistream",
        "multi_stream",
        "eegattentiontransformer",
        "eeg_attention_transformer",
        "attentiontransformer",
    ]

    if model_name not in valid_names:
        raise ValueError(
            "Esta versión está preparada para "
            "model_name='MultiStream' o 'EEGAttentionTransformer'."
        )

    # Importante: usa la semilla que le mande la rutina: seed + fold
    set_seed(kwargs.get("seed", 42))

    base_model = EEGAttentionTransformer(
        freq_shape=kwargs.get("freq_shape", (20, 1)),
        temp_shape=kwargs.get("temp_shape", (10, 1)),
        spat_shape=kwargs.get("spat_shape", (19, 1)),

        d_model=kwargs.get("d_model", 64),
        num_layers=kwargs.get("num_layers", 2),
        num_heads=kwargs.get("num_heads", 4),
        d_ff=kwargs.get("d_ff", 128),
        num_classes=kwargs.get("nb_classes", 2),
    )

    return_dict = kwargs.get("return_dict", True)

    if return_dict:
        return MultiStreamReturnDictWrapper(base_model)

    return base_model

In [10]:
# =========================================================
# TRAIN L24O / CV FIJO - PYTORCH
# Compatible con:
#   - Modelos de una entrada: X -> (N, C, T)
#   - Modelos multi-stream: X -> (freq, temp, spat)
# =========================================================

import os
import json
import copy
import numpy as np
import torch

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split


# =========================================================
# AUXILIAR: convertir splits de folds a índices de ventanas
# =========================================================

def _split_to_window_indices(split, sbjs):
    """
    Convierte un split del fold a índices de ventanas.

    Soporta:
    1) índices de ventanas:
        [0, 1, 2, 3, ...]

    2) nombres de sujetos:
        ["v1p", "v2p", ...]

    3) índices de sujetos:
        [0, 1, 2, ...]

    4) máscara booleana:
        [True, False, ...]
    """

    groups = np.asarray(sbjs)
    n_samples = len(groups)

    unique_subjects = np.array(list(dict.fromkeys(groups)))
    n_subjects = len(unique_subjects)

    arr = np.asarray(split)

    # Caso máscara booleana de ventanas
    if arr.dtype == bool:
        if len(arr) != n_samples:
            raise ValueError(
                f"Máscara booleana inválida: len={len(arr)}, esperado={n_samples}"
            )
        return np.where(arr)[0].astype(np.int64)

    # Caso vacío
    if arr.size == 0:
        return np.array([], dtype=np.int64)

    # Caso nombres de sujetos
    if arr.dtype.kind in {"U", "S", "O"}:
        subject_set = set(arr.tolist())
        idx = np.where(np.isin(groups, list(subject_set)))[0]
        return idx.astype(np.int64)

    # Caso numérico
    if arr.dtype.kind in {"i", "u"}:
        arr = arr.astype(np.int64)

        # Heurística:
        # si parece lista de sujetos, no de ventanas
        if len(arr) <= n_subjects and np.max(arr) < n_subjects:
            selected_subjects = unique_subjects[arr]
            idx = np.where(np.isin(groups, selected_subjects))[0]
            return idx.astype(np.int64)

        # Si no, lo tomamos como índices de ventanas
        if np.max(arr) >= n_samples:
            raise ValueError(
                f"Índice fuera de rango: max={np.max(arr)}, n_samples={n_samples}"
            )

        return arr.astype(np.int64)

    raise ValueError(f"No se pudo interpretar split con dtype={arr.dtype}")


# =========================================================
# AUXILIAR: extraer train/val/test de cada fold
# =========================================================

def _resolve_fold_indices(fold, sbjs, y_binary, seed=42, fold_id=1):
    """
    Devuelve:
        train_idx, val_idx, test_idx

    Soporta folds como:
        dict con keys train/val/test
        tuple/list de longitud 2: (train, test)
        tuple/list de longitud 3: (train, val, test)

    Si no existe val, crea validación desde train por sujetos.
    """

    groups = np.asarray(sbjs)

    train_split = None
    val_split = None
    test_split = None

    # -----------------------------------------------------
    # Caso fold tipo diccionario
    # -----------------------------------------------------
    if isinstance(fold, dict):
        train_keys = [
            "train_idx", "train_indices", "idx_train",
            "train", "train_subjects", "subjects_train"
        ]

        val_keys = [
            "val_idx", "valid_idx", "validation_idx",
            "idx_val", "idx_valid",
            "val", "valid", "validation",
            "val_subjects", "valid_subjects", "subjects_val"
        ]

        test_keys = [
            "test_idx", "test_indices", "idx_test",
            "test", "test_subjects", "subjects_test"
        ]

        for k in train_keys:
            if k in fold:
                train_split = fold[k]
                break

        for k in val_keys:
            if k in fold:
                val_split = fold[k]
                break

        for k in test_keys:
            if k in fold:
                test_split = fold[k]
                break

    # -----------------------------------------------------
    # Caso fold tipo tuple/list
    # -----------------------------------------------------
    elif isinstance(fold, (tuple, list)):
        if len(fold) == 2:
            train_split, test_split = fold
            val_split = None

        elif len(fold) == 3:
            train_split, val_split, test_split = fold

        else:
            raise ValueError(
                f"Fold con longitud no soportada: len(fold)={len(fold)}"
            )

    else:
        raise ValueError(f"Tipo de fold no soportado: {type(fold)}")

    if train_split is None or test_split is None:
        raise ValueError(
            "No se pudieron identificar train/test en el fold. "
            f"Fold recibido: {fold}"
        )

    train_idx = _split_to_window_indices(train_split, sbjs)
    test_idx = _split_to_window_indices(test_split, sbjs)

    # -----------------------------------------------------
    # Si existe val en el fold
    # -----------------------------------------------------
    if val_split is not None:
        val_idx = _split_to_window_indices(val_split, sbjs)
        return train_idx, val_idx, test_idx

    # -----------------------------------------------------
    # Si NO existe val, se crea desde train por sujeto
    # -----------------------------------------------------
    train_subjects = np.array(list(dict.fromkeys(groups[train_idx])))

    subject_labels = []

    for subject in train_subjects:
        subject_window_idx = np.where(groups == subject)[0]
        subject_labels.append(y_binary[subject_window_idx[0]])

    subject_labels = np.asarray(subject_labels, dtype=np.int64)

    # Intentar split estratificado por sujeto
    try:
        stratify_arg = subject_labels

        # Si alguna clase tiene menos de 2 sujetos, no se puede estratificar
        _, class_counts = np.unique(subject_labels, return_counts=True)
        if np.min(class_counts) < 2:
            stratify_arg = None

        train_subjects_new, val_subjects = train_test_split(
            train_subjects,
            test_size=0.2,
            random_state=seed + fold_id,
            shuffle=True,
            stratify=stratify_arg,
        )

    except Exception:
        train_subjects_new, val_subjects = train_test_split(
            train_subjects,
            test_size=0.2,
            random_state=seed + fold_id,
            shuffle=True,
            stratify=None,
        )

    train_idx_new = np.where(np.isin(groups, train_subjects_new))[0]
    val_idx = np.where(np.isin(groups, val_subjects))[0]

    return train_idx_new.astype(np.int64), val_idx.astype(np.int64), test_idx.astype(np.int64)


# =========================================================
# AUXILIAR: crear datasets
# =========================================================

def _make_dataset_from_X(X, y, indices):
    """
    Crea dataset según el tipo de X.

    Caso 1:
        X es ndarray: modelo de una entrada.

    Caso 2:
        X es tuple: (freq, temp, spat), modelo multi-stream.
    """

    if isinstance(X, tuple) and len(X) == 3:
        freq, temp, spat = X

        dataset = MultiStreamEEGDataset(
            freq=freq[indices],
            temp=temp[indices],
            spat=spat[indices],
            y=y[indices],
        )

        return dataset

    else:
        dataset = TensorDataset(
            torch.tensor(X[indices], dtype=torch.float32),
            torch.tensor(y[indices], dtype=torch.float32),
        )

        return dataset


# =========================================================
# AUXILIAR: promedio de métricas
# =========================================================

def _mean_scalar_metrics(metrics_list):
    """
    Promedia solo métricas escalares.
    Ignora arreglos como y_true, y_pred, y_prob.
    """

    if len(metrics_list) == 0:
        return None

    scalar_keys = []

    for k, v in metrics_list[0].items():
        if isinstance(v, (int, float, np.integer, np.floating)):
            scalar_keys.append(k)

    mean_metrics = {}

    for k in scalar_keys:
        vals = []

        for m in metrics_list:
            if k in m and isinstance(m[k], (int, float, np.integer, np.floating)):
                vals.append(float(m[k]))

        if len(vals) > 0:
            mean_metrics[k] = float(np.nanmean(vals))

    return mean_metrics


# =========================================================
# AUXILIAR: guardar JSON sin arrays grandes
# =========================================================

def _metrics_to_jsonable(metrics):
    """
    Convierte métricas a formato serializable.
    Omite y_true, y_pred, y_prob para JSON.
    """

    if metrics is None:
        return None

    out = {}

    for k, v in metrics.items():
        if k in ["y_true", "y_pred", "y_prob"]:
            continue

        if isinstance(v, (int, float, np.integer, np.floating)):
            out[k] = float(v)
        else:
            out[k] = str(v)

    return out


# =========================================================
# FUNCIÓN PRINCIPAL DE ENTRENAMIENTO
# =========================================================

def train_L24O_cv_fixed(
    model_name,
    X,
    y,
    sbjs,
    folds,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    save_root="./results",
    run_name="run_fixed",
    cache_model_format="weights",
    force_retrain=True,
    evaluate_test=True,
):
    """
    Entrenamiento CV fijo en PyTorch.

    Compatible con:
        - X ndarray: modelo de una entrada.
        - X tuple: (freq, temp, spat), modelo multi-stream.

    Requiere que existan:
        - build_eeg_model
        - _run_epoch_pytorch
        - _evaluate_pytorch_classifier
        - _build_optimizer_and_scheduler
        - set_seed
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    set_seed(seed)

    run_dir = os.path.join(save_root, run_name)
    os.makedirs(run_dir, exist_ok=True)

    y = np.asarray(y)
    y_binary = onehot_to_binary_labels(y)
    groups = np.asarray(sbjs)

    fold_results = []
    val_metrics_all = []
    test_metrics_all = []

    print("\n" + "=" * 80)
    print(f"INICIANDO ENTRENAMIENTO CV FIJO - {model_name}")
    print("=" * 80)
    print("Device:", device)
    print("Run dir:", run_dir)
    print("N muestras:", len(y))
    print("N sujetos:", len(set(sbjs)))
    print("N folds:", len(folds))

    for fold_id, fold in enumerate(folds, start=1):
        print("\n" + "-" * 80)
        print(f"FOLD {fold_id}/{len(folds)}")
        print("-" * 80)

        fold_dir = os.path.join(run_dir, f"fold_{fold_id:02d}")
        os.makedirs(fold_dir, exist_ok=True)

        best_model_path = os.path.join(fold_dir, "best_model_weights.pt")
        fold_json_path = os.path.join(fold_dir, "fold_results.json")

        # -------------------------------------------------
        # Resolver índices
        # -------------------------------------------------

        train_idx, val_idx, test_idx = _resolve_fold_indices(
            fold=fold,
            sbjs=sbjs,
            y_binary=y_binary,
            seed=seed,
            fold_id=fold_id,
        )

        print("Train windows:", len(train_idx))
        print("Val windows  :", len(val_idx))
        print("Test windows :", len(test_idx))

        print("Train subjects:", len(set(groups[train_idx])))
        print("Val subjects  :", len(set(groups[val_idx])))
        print("Test subjects :", len(set(groups[test_idx])))

        print("Train classes:", np.unique(y_binary[train_idx], return_counts=True))
        print("Val classes  :", np.unique(y_binary[val_idx], return_counts=True))
        print("Test classes :", np.unique(y_binary[test_idx], return_counts=True))

        # -------------------------------------------------
        # Datasets y loaders
        # -------------------------------------------------

        train_dataset = _make_dataset_from_X(X, y, train_idx)
        val_dataset = _make_dataset_from_X(X, y, val_idx)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        if evaluate_test:
            test_dataset = _make_dataset_from_X(X, y, test_idx)

            test_loader = DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )
        else:
            test_loader = None

        # -------------------------------------------------
        # Crear modelo
        # -------------------------------------------------

        model = build_eeg_model(
            model_name,
            seed=seed + fold_id,
            **model_args,
        ).to(device)

        loss_fn = compile_cfg["loss_fn"]

        optimizer, scheduler = _build_optimizer_and_scheduler(
            model=model,
            compile_args=compile_cfg,
        )

        patience = compile_cfg.get("early_stopping_patience", 25)
        min_delta = compile_cfg.get("min_delta", 1e-4)

        # -------------------------------------------------
        # Si ya existe y no se fuerza reentrenamiento
        # -------------------------------------------------

        if (not force_retrain) and os.path.exists(best_model_path):
            print("Cargando modelo existente:", best_model_path)
            model.load_state_dict(torch.load(best_model_path, map_location=device))

            best_epoch = None
            best_val_metrics = _evaluate_pytorch_classifier(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

        else:
            # -------------------------------------------------
            # Entrenamiento
            # -------------------------------------------------

            best_val_loss = np.inf
            best_state = None
            best_epoch = 0
            best_val_metrics = None
            patience_counter = 0

            history = []

            for epoch in range(1, epochs + 1):
                train_loss = _run_epoch_pytorch(
                    model=model,
                    loader=train_loader,
                    optimizer=optimizer,
                    loss_fn=loss_fn,
                    device=device,
                )

                val_metrics = _evaluate_pytorch_classifier(
                    model=model,
                    loader=val_loader,
                    loss_fn=loss_fn,
                    device=device,
                )

                val_loss = val_metrics["loss"]

                if scheduler is not None:
                    scheduler.step(val_loss)

                history.append({
                    "epoch": epoch,
                    "train_loss": float(train_loss),
                    "val_loss": float(val_loss),
                    "val_accuracy": float(val_metrics["accuracy"]),
                    "val_balanced_accuracy": float(val_metrics["balanced_accuracy"]),
                    "val_auc": float(val_metrics["auc"]) if not np.isnan(val_metrics["auc"]) else None,
                })

                print(
                    f"Epoch {epoch:03d}/{epochs} | "
                    f"train_loss={train_loss:.4f} | "
                    f"val_loss={val_loss:.4f} | "
                    f"val_acc={val_metrics['accuracy']:.4f} | "
                    f"val_bacc={val_metrics['balanced_accuracy']:.4f} | "
                    f"val_auc={val_metrics['auc']:.4f}"
                )

                # Early stopping por val_loss
                if val_loss < best_val_loss - min_delta:
                    best_val_loss = val_loss
                    best_state = copy.deepcopy(model.state_dict())
                    best_epoch = epoch
                    best_val_metrics = val_metrics
                    patience_counter = 0

                    torch.save(best_state, best_model_path)

                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    print(
                        f"Early stopping en epoch {epoch}. "
                        f"Mejor epoch: {best_epoch}"
                    )
                    break

            # Cargar mejor estado
            if best_state is not None:
                model.load_state_dict(best_state)
            elif os.path.exists(best_model_path):
                model.load_state_dict(torch.load(best_model_path, map_location=device))

            # Guardar history
            history_path = os.path.join(fold_dir, "history.json")

            with open(history_path, "w") as f:
                json.dump(history, f, indent=4)

        # -------------------------------------------------
        # Evaluación final en val con mejor modelo
        # -------------------------------------------------

        final_val_metrics = _evaluate_pytorch_classifier(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        val_metrics_all.append(final_val_metrics)

        # -------------------------------------------------
        # Evaluación test
        # -------------------------------------------------

        if evaluate_test and test_loader is not None:
            final_test_metrics = _evaluate_pytorch_classifier(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            test_metrics_all.append(final_test_metrics)
        else:
            final_test_metrics = None

        # -------------------------------------------------
        # Guardar resultado del fold
        # -------------------------------------------------

        fold_result = {
            "fold": fold_id,
            "best_epoch": best_epoch,
            "n_train_windows": int(len(train_idx)),
            "n_val_windows": int(len(val_idx)),
            "n_test_windows": int(len(test_idx)),
            "n_train_subjects": int(len(set(groups[train_idx]))),
            "n_val_subjects": int(len(set(groups[val_idx]))),
            "n_test_subjects": int(len(set(groups[test_idx]))),
            "val_metrics": final_val_metrics,
            "test_metrics": final_test_metrics,
            "model_path": best_model_path,
        }

        fold_results.append(fold_result)

        fold_result_json = {
            "fold": fold_id,
            "best_epoch": best_epoch,
            "n_train_windows": int(len(train_idx)),
            "n_val_windows": int(len(val_idx)),
            "n_test_windows": int(len(test_idx)),
            "n_train_subjects": int(len(set(groups[train_idx]))),
            "n_val_subjects": int(len(set(groups[val_idx]))),
            "n_test_subjects": int(len(set(groups[test_idx]))),
            "val_metrics": _metrics_to_jsonable(final_val_metrics),
            "test_metrics": _metrics_to_jsonable(final_test_metrics),
            "model_path": best_model_path,
        }

        with open(fold_json_path, "w") as f:
            json.dump(fold_result_json, f, indent=4)

        print("\nResumen fold:")
        print(f"  Val accuracy          : {final_val_metrics['accuracy']:.4f}")
        print(f"  Val balanced accuracy : {final_val_metrics['balanced_accuracy']:.4f}")
        print(f"  Val AUC               : {final_val_metrics['auc']:.4f}")

        if final_test_metrics is not None:
            print(f"  Test accuracy         : {final_test_metrics['accuracy']:.4f}")
            print(f"  Test balanced accuracy: {final_test_metrics['balanced_accuracy']:.4f}")
            print(f"  Test AUC              : {final_test_metrics['auc']:.4f}")

    # =====================================================
    # Promedios globales
    # =====================================================

    mean_val_metrics = _mean_scalar_metrics(val_metrics_all)
    mean_metrics = _mean_scalar_metrics(test_metrics_all) if evaluate_test else None

    results = {
        "model_name": model_name,
        "run_name": run_name,
        "run_dir": run_dir,
        "seed": seed,
        "epochs": epochs,
        "batch_size": batch_size,
        "fold_results": fold_results,
        "mean_val_metrics": mean_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_accuracy": mean_val_metrics.get("accuracy") if mean_val_metrics else None,
        "mean_val_balanced_accuracy": mean_val_metrics.get("balanced_accuracy") if mean_val_metrics else None,
        "mean_accuracy": mean_metrics.get("accuracy") if mean_metrics else None,
    }

    summary_path = os.path.join(run_dir, "cv_summary.json")

    summary_json = {
        "model_name": model_name,
        "run_name": run_name,
        "run_dir": run_dir,
        "seed": seed,
        "epochs": epochs,
        "batch_size": batch_size,
        "mean_val_metrics": mean_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_accuracy": results["mean_val_accuracy"],
        "mean_val_balanced_accuracy": results["mean_val_balanced_accuracy"],
        "mean_accuracy": results["mean_accuracy"],
    }

    with open(summary_path, "w") as f:
        json.dump(summary_json, f, indent=4)

    print("\n" + "=" * 80)
    print("CV FINALIZADO")
    print("=" * 80)

    print("\nPromedio validación:")
    for k, v in mean_val_metrics.items():
        print(f"  {k}: {v:.4f}")

    if mean_metrics is not None:
        print("\nPromedio test:")
        for k, v in mean_metrics.items():
            print(f"  {k}: {v:.4f}")

    return results

In [11]:
# =========================================================
# UTILIDADES PARA ETIQUETAS
# =========================================================

def ensure_onehot_labels(y):
    """
    Asegura que y quede en formato one-hot:

    Clase 0 -> [1, 0]
    Clase 1 -> [0, 1]

    Sirve si quieres mantener compatibilidad con rutinas tipo Keras/T-GARNet.
    """
    y = np.asarray(y)

    # Ya viene one-hot: shape (N, 2)
    if y.ndim == 2 and y.shape[1] == 2:
        return y.astype(np.float32)

    # Viene como etiquetas binarias: shape (N,)
    if y.ndim == 1:
        y = y.astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    # Viene como columna: shape (N, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.squeeze(1).astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    raise ValueError(f"Formato de y no soportado para one-hot: shape={y.shape}")


def onehot_to_binary_labels(y):
    """
    Convierte y a etiquetas binarias 0/1.

    Sirve para:
    - StratifiedGroupKFold
    - métricas
    - agrupación por sujeto
    - reportes finales
    """
    y = np.asarray(y)

    if y.ndim == 1:
        return y.astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1).astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 2:
        return np.argmax(y, axis=1).astype(np.int64)

    raise ValueError(f"Formato de y no soportado: shape={y.shape}")


# =========================================================
# DATASET PARA MODELOS MULTI-STREAM
# =========================================================

class MultiStreamEEGDataset(Dataset):
    """
    Dataset para el modelo MultiStream Transformer.

    Devuelve:
        freq : (20, 1)
        temp : (10, 1)
        spat : (C, 1)
        y    : one-hot o etiqueta, según lo que entregues
    """

    def __init__(self, freq, temp, spat, y):
        self.freq = torch.tensor(freq, dtype=torch.float32)
        self.temp = torch.tensor(temp, dtype=torch.float32)
        self.spat = torch.tensor(spat, dtype=torch.float32)

        # Se deja como float porque la rutina admite y one-hot
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.freq[idx],
            self.temp[idx],
            self.spat[idx],
            self.y[idx],
        )


# =========================================================
# LOSS PARA MODELOS QUE DEVUELVEN DICCIONARIO
# =========================================================

class ClassificationLossFromLogits(nn.Module):
    """
    Loss para modelos que devuelven:

        outputs["logits"]         -> salida sin softmax
        outputs["out_activation"] -> probabilidades softmax

    Puede recibir y_true como:
        - one-hot: (B, 2)
        - entero:  (B,)
    """

    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, outputs, y_true):
        logits = outputs["logits"]

        if y_true.ndim == 2:
            y_true = torch.argmax(y_true, dim=1)

        y_true = y_true.long()

        return self.ce(logits, y_true)


# =========================================================
# WRAPPER PARA MULTI-STREAM CON RETURN_DICT
# =========================================================

class MultiStreamReturnDictWrapper(nn.Module):
    """
    Envuelve el modelo MultiStream para que sea compatible con rutinas
    que esperan outputs["out_activation"].

    El modelo base devuelve logits.
    El wrapper devuelve:

        {
            "logits": logits,
            "out_activation": softmax(logits)
        }
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, freq, temp, spat):
        logits = self.model(freq, temp, spat)

        out_activation = torch.softmax(logits, dim=1)

        return {
            "logits": logits,
            "out_activation": out_activation,
        }


# =========================================================
# OPTIMIZER / SCHEDULER
# =========================================================

def _build_optimizer_and_scheduler(model, compile_args):
    """
    Construye optimizer y scheduler desde compile_args.

    Ejemplo:

    compile_args = {
        "optimizer_name": "adam",
        "optimizer_params": {
            "lr": 1e-3,
            "weight_decay": 0.0,
        },
        "scheduler_name": "ReduceLROnPlateau",
        "scheduler_params": {
            "mode": "min",
            "factor": 0.5,
            "patience": 10,
            "min_lr": 1e-6,
        },
    }
    """

    optimizer_name = compile_args["optimizer_name"].lower()
    optimizer_params = deepcopy(compile_args["optimizer_params"])

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), **optimizer_params)

    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), **optimizer_params)

    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), **optimizer_params)

    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_name}")

    scheduler_name = compile_args.get("scheduler_name", None)
    scheduler = None

    if scheduler_name is not None:
        scheduler_name = scheduler_name.lower()
        scheduler_params = deepcopy(compile_args.get("scheduler_params", {}))

        if scheduler_name == "reducelronplateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                **scheduler_params,
            )

        else:
            raise ValueError(f"Scheduler no soportado: {scheduler_name}")

    return optimizer, scheduler


# =========================================================
# RESTRICCIONES TIPO KERAS DESPUÉS DEL OPTIMIZER.STEP()
# =========================================================

def apply_model_constraints_after_step(model):
    """
    Aplica restricciones después de optimizer.step().

    Para ShallowConvNetTorch:
        usa model.apply_max_norm()

    Para EEGNetTorch u otros modelos:
        intenta usar model.apply_constraints()

    Para MultiStream Transformer:
        normalmente no hace nada.
    """

    if hasattr(model, "apply_max_norm"):
        model.apply_max_norm()

    elif hasattr(model, "apply_constraints"):
        model.apply_constraints()

    elif hasattr(model, "model"):
        # Por si el modelo está envuelto en un wrapper
        if hasattr(model.model, "apply_max_norm"):
            model.model.apply_max_norm()

        elif hasattr(model.model, "apply_constraints"):
            model.model.apply_constraints()


# =========================================================
# AUXILIAR PARA MOVER BATCH AL DEVICE
# =========================================================

def unpack_batch_to_device(batch, device):
    """
    Soporta dos formatos de batch.

    1) Modelo de una entrada:
        batch = (xb, yb)

        Se llama:
        outputs = model(xb)

    2) Modelo multi-stream:
        batch = (freq_b, temp_b, spat_b, yb)

        Se llama:
        outputs = model(freq_b, temp_b, spat_b)
    """

    if len(batch) == 2:
        xb, yb = batch

        xb = xb.to(device=device, dtype=torch.float32)

        # Puede ser one-hot float o entero.
        # Se deja float para ser compatible con loss que convierten internamente.
        yb = yb.to(device=device)

        return (xb,), yb

    elif len(batch) == 4:
        freq_b, temp_b, spat_b, yb = batch

        freq_b = freq_b.to(device=device, dtype=torch.float32)
        temp_b = temp_b.to(device=device, dtype=torch.float32)
        spat_b = spat_b.to(device=device, dtype=torch.float32)

        # Puede ser one-hot float o entero.
        yb = yb.to(device=device)

        return (freq_b, temp_b, spat_b), yb

    else:
        raise ValueError(
            f"Formato de batch no soportado. "
            f"Se esperaba len(batch)=2 o len(batch)=4, pero llegó len={len(batch)}"
        )


# =========================================================
# UNA ÉPOCA DE ENTRENAMIENTO
# =========================================================

def _run_epoch_pytorch(model, loader, optimizer, loss_fn, device):
    """
    Ejecuta una época de entrenamiento.

    Compatible con:

    1) Modelos de una entrada:
        outputs = model(xb)

    2) Modelos multi-stream:
        outputs = model(freq_b, temp_b, spat_b)

    El modelo debe devolver un diccionario con al menos:

        outputs["out_activation"]

    Si se usa ClassificationLossFromLogits, también debe devolver:

        outputs["logits"]
    """

    model.train()

    running_loss = 0.0
    n_samples = 0

    for batch in loader:
        inputs, yb = unpack_batch_to_device(batch, device)

        optimizer.zero_grad()

        outputs = model(*inputs)

        loss = loss_fn(outputs, yb)

        loss.backward()
        optimizer.step()

        # Para ShallowConvNet aplica max_norm.
        # Para MultiStream no hace nada si no existe el método.
        apply_model_constraints_after_step(model)

        batch_size_curr = yb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

    epoch_loss = running_loss / max(n_samples, 1)

    return epoch_loss


# =========================================================
# EVALUACIÓN
# =========================================================

@torch.no_grad()
def _evaluate_pytorch_classifier(model, loader, loss_fn, device):
    """
    Evalúa el modelo y calcula métricas.

    Compatible con modelos que devuelven:

        outputs["out_activation"] = probabilidades softmax, shape (B, 2)

    Soporta:

        - entrada única: xb
        - multi-stream: freq, temp, spat
    """

    model.eval()

    running_loss = 0.0
    n_samples = 0

    all_probs = []
    all_preds = []
    all_true = []

    for batch in loader:
        inputs, yb = unpack_batch_to_device(batch, device)

        outputs = model(*inputs)

        probs_2c = outputs["out_activation"]

        loss = loss_fn(outputs, yb)

        probs_pos = probs_2c[:, 1]
        preds = torch.argmax(probs_2c, dim=1)

        # yb puede venir one-hot o como etiqueta entera
        if yb.ndim == 2:
            y_true = torch.argmax(yb, dim=1)
        else:
            y_true = yb.long()

        batch_size_curr = yb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

        all_probs.append(probs_pos.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(y_true.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs).astype(np.float32)
    y_pred = np.concatenate(all_preds).astype(np.int64)
    y_true = np.concatenate(all_true).astype(np.int64)

    mean_loss = running_loss / max(n_samples, 1)

    try:
        auc_value = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_value = np.nan

    metrics = {
        "loss": mean_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "recall": recall_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "auc": auc_value,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    return metrics


# =========================================================
# CONFIGURACIÓN DE COMPILACIÓN PARA MULTI-STREAM
# =========================================================

compile_cfg = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": "ReduceLROnPlateau",
    "scheduler_params": {
        "mode": "min",
        "factor": 0.5,
        "patience": 10,
        "min_lr": 1e-6,
    },
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": ClassificationLossFromLogits(),
}

In [12]:
# =========================================================
# IMPORTS
# =========================================================

import pickle
import os
import json
import warnings

import numpy as np
import torch

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

model_name = "MultiStream"
db = "TDAH"
seed = 42

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"


# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# SEMILLA
# =========================================================

set_seed(seed)


# =========================================================
# RUTA DIRECTA DE GUARDADO
# =========================================================

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_multistream_tdah_fixed_seed42"
os.makedirs(SAVE_ROOT, exist_ok=True)

run_name = f"{model_name}_{db}_fixed_seed{seed}"

RESULTS_JSON = os.path.join(SAVE_ROOT, "summary_results.json")
RESULTS_PKL = os.path.join(SAVE_ROOT, "summary_results.pkl")


# =========================================================
# CARGAR FOLDS
# =========================================================

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)


# =========================================================
# CARGAR DATA CRUDA SEGMENTADA
# =========================================================
# X_raw: (N, 19, 512)
# y    : puede venir one-hot o binaria según tu función get_segmented_data()

X_raw, y, sbjs, _ = get_segmented_data()

print("=" * 80)
print("DATA CARGADA")
print("=" * 80)
print("X_raw:", X_raw.shape, X_raw.dtype)
print("y:", y.shape, y.dtype)
print("Número de sujetos:", len(set(sbjs)))
print("Número de folds:", len(folds))


# =========================================================
# ASEGURAR FORMATO DE ETIQUETAS
# =========================================================
# Para la rutina general dejamos y en one-hot.
# ClassificationLossFromLogits internamente convierte one-hot a etiquetas 0/1.

y_onehot = ensure_onehot_labels(y)
y_binary = onehot_to_binary_labels(y_onehot)

print("\nEtiquetas:")
print("y_onehot:", y_onehot.shape, y_onehot.dtype)
print("y_binary:", y_binary.shape, y_binary.dtype)
print("Clases:", np.unique(y_binary, return_counts=True))


# =========================================================
# PREPARAR STREAMS PARA EL MODELO MULTI-STREAM
# =========================================================
# freq: (N, 20, 1)
# temp: (N, 10, 1)
# spat: (N, 19, 1)

freq, temp, spat = prepare_streams_4s(
    X_raw,
    fs=128,
    n_win=10,
)

print("\nSTREAMS PREPARADOS")
print("=" * 80)
print("freq:", freq.shape, freq.dtype)
print("temp:", temp.shape, temp.dtype)
print("spat:", spat.shape, spat.dtype)


# =========================================================
# X PARA LA RUTINA MULTI-STREAM
# =========================================================
# Importante:
# train_L24O_cv_fixed debe saber que si X es una tupla,
# entonces debe indexar cada stream por separado:
#
# freq_train = X[0][train_idx]
# temp_train = X[1][train_idx]
# spat_train = X[2][train_idx]

X = (freq, temp, spat)


# =========================================================
# PARÁMETROS FIJOS MULTI-STREAM TRANSFORMER
# =========================================================

model_args = {
    "nb_classes": 2,

    "freq_shape": (20, 1),
    "temp_shape": (10, 1),
    "spat_shape": (19, 1),

    "d_model": 64,
    "num_layers": 2,
    "num_heads": 4,
    "d_ff": 128,

    "return_dict": True,
}


# =========================================================
# CONFIGURACIÓN DE COMPILACIÓN PYTORCH
# BASELINE FIJO SIN SCHEDULER
# =========================================================

compile_cfg = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": ClassificationLossFromLogits(),
}


# Versión serializable para guardar en JSON
compile_cfg_json = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": "ClassificationLossFromLogits",
}


# =========================================================
# ENTRENAR MULTI-STREAM FIJO
# =========================================================

results = train_L24O_cv_fixed(
    model_name=model_name,

    # Ahora X es una tupla:
    # X = (freq, temp, spat)
    X=X,

    # Dejamos y en one-hot para compatibilidad con la rutina.
    y=y_onehot,

    sbjs=sbjs,
    folds=folds,

    model_args=model_args,
    compile_cfg=compile_cfg,

    epochs=epochs,
    batch_size=batch_size,
    seed=seed,

    save_root=SAVE_ROOT,
    run_name=run_name,

    cache_model_format=save_format,
    force_retrain=force_retrain,
    evaluate_test=True,
)


# =========================================================
# GUARDAR RESULTADOS GLOBALES
# =========================================================

with open(RESULTS_PKL, "wb") as f:
    pickle.dump(results, f)


json_results = {
    "model_name": model_name,
    "db": db,
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,

    "model_args": model_args,
    "compile_cfg": compile_cfg_json,

    "input_shapes": {
        "X_raw": list(X_raw.shape),
        "freq": list(freq.shape),
        "temp": list(temp.shape),
        "spat": list(spat.shape),
        "y_onehot": list(y_onehot.shape),
    },

    "mean_accuracy": results.get("mean_accuracy"),
    "mean_val_accuracy": results.get("mean_val_accuracy"),
    "mean_val_balanced_accuracy": results.get("mean_val_balanced_accuracy"),
    "mean_metrics": results.get("mean_metrics"),
    "mean_val_metrics": results.get("mean_val_metrics"),
    "run_dir": results.get("run_dir"),
}


with open(RESULTS_JSON, "w") as f:
    json.dump(json_results, f, indent=4)


# =========================================================
# RESUMEN FINAL
# =========================================================

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINALIZADO - MULTI-STREAM TRANSFORMER FIJO")
print("=" * 80)
print(f"Carpeta principal       : {SAVE_ROOT}")
print(f"Run name                : {run_name}")
print(f"Seed                    : {seed}")
print(f"Resultados PKL          : {RESULTS_PKL}")
print(f"Resultados JSON         : {RESULTS_JSON}")

print("\nValidación:")
print(f"  mean_val_accuracy          : {results['mean_val_accuracy']:.4f}")
print(f"  mean_val_balanced_accuracy : {results['mean_val_balanced_accuracy']:.4f}")

if results["mean_metrics"] is not None:
    print("\nTest:")
    for k, v in results["mean_metrics"].items():
        if isinstance(v, (int, float, np.floating)):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

Using device: cuda
Sujeto eliminado explícitamente: v36p
Sujetos TDAH encontrados: 60
Sujetos Control encontrados: 60
Sujetos totales encontrados: 120


DATA CARGADA
X_raw: (8213, 19, 512) float32
y: (8213, 2) float32
Número de sujetos: 120
Número de folds: 5

Etiquetas:
y_onehot: (8213, 2) float32
y_binary: (8213,) int64
Clases: (array([0, 1]), array([3657, 4556]))

STREAMS PREPARADOS
freq: (8213, 20, 1) float32
temp: (8213, 10, 1) float32
spat: (8213, 19, 1) float32

INICIANDO ENTRENAMIENTO CV FIJO - MultiStream
Device: cuda
Run dir: /home/usuario-utp/Desktop/Yessica Alejandra/resultados_multistream_tdah_fixed_seed42/MultiStream_TDAH_fixed_seed42
N muestras: 8213
N sujetos: 120
N folds: 5

--------------------------------------------------------------------------------
FOLD 1/5
--------------------------------------------------------------------------------
Train windows: 4977
Val windows  : 1574
Test windows : 1662
Train subjects: 76
Val subjects  : 20
Test subjects : 24
Train classes: (array([0, 1]), array([2231, 2746]))
Val classes  : (array([0, 1]), array([683, 891]))
Test classes : (array([0, 1]), array([743, 919]))
Epoch 001/10

In [13]:
# =========================================================
# REPETICIONES 10 SEEDS - MULTI-STREAM TRANSFORMER
# =========================================================

import os
import json
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")


# =========================================================
# 1) CONFIGURACIÓN GENERAL
# =========================================================

model_name = "MultiStream"
db = "TDAH"

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_multistream_tdah_repeated_no_scheduler"
os.makedirs(SAVE_ROOT, exist_ok=True)

RESULTS_CSV = os.path.join(SAVE_ROOT, "repeated_test_results.csv")
RESULTS_JSON = os.path.join(SAVE_ROOT, "repeated_test_summary.json")

# 10 repeticiones -> cada seed base corre los 5 folds
seeds = list(range(10))

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"


# =========================================================
# 2) DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# 3) PARÁMETROS FIJOS MULTI-STREAM TRANSFORMER
# =========================================================

model_args = {
    "nb_classes": 2,

    "freq_shape": (20, 1),
    "temp_shape": (10, 1),
    "spat_shape": (19, 1),

    "d_model": 64,
    "num_layers": 2,
    "num_heads": 4,
    "d_ff": 128,

    "return_dict": True,
}


# =========================================================
# 4) CONFIGURACIÓN DE COMPILACIÓN PYTORCH
# BASELINE FIJO SIN SCHEDULER
# =========================================================

compile_cfg = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": ClassificationLossFromLogits(),
}


# Versión serializable para JSON
compile_cfg_json = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": "ClassificationLossFromLogits",
}


# =========================================================
# 5) FUNCIONES AUXILIARES
# =========================================================

def set_all_seeds(seed: int):
    """
    Seed base de cada repetición.
    Dentro de train_L24O_cv_fixed se usará seed + fold.
    """
    set_seed(seed)


def summarize_metric(values):
    arr = np.array(values, dtype=float)

    return {
        "mean": float(np.nanmean(arr)),
        "std_population": float(np.nanstd(arr, ddof=0)),
        "std_sample": float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0,
        "n": int(len(arr)),
    }


# =========================================================
# 6) CARGAR DATOS Y FOLDS
# =========================================================

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)


# ---------------------------------------------------------
# Cargar EEG crudo segmentado
# ---------------------------------------------------------
# X_raw: (N, 19, 512)
# y    : puede venir como 0/1 o one-hot, dependiendo de tu get_segmented_data()

X_raw, y, sbjs, _ = get_segmented_data()


# ---------------------------------------------------------
# Asegurar etiquetas one-hot para compatibilidad
# ---------------------------------------------------------

y_onehot = ensure_onehot_labels(y)
y_binary = onehot_to_binary_labels(y_onehot)


# ---------------------------------------------------------
# Preparar streams
# ---------------------------------------------------------
# freq: (N, 20, 1)
# temp: (N, 10, 1)
# spat: (N, 19, 1)

freq, temp, spat = prepare_streams_4s(
    X_raw,
    fs=128,
    n_win=10,
)

# X multi-stream para la rutina
X = (freq, temp, spat)


# =========================================================
# 7) MOSTRAR CONFIGURACIÓN
# =========================================================

print("=" * 90)
print("CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - MULTI-STREAM TRANSFORMER")
print("=" * 90)

print("model_args:")
for k, v in model_args.items():
    print(f"  {k}: {v}")

print("\ncompile_cfg:")
for k, v in compile_cfg_json.items():
    print(f"  {k}: {v}")

print("\nDatos:")
print("X_raw:", X_raw.shape, X_raw.dtype)
print("freq :", freq.shape, freq.dtype)
print("temp :", temp.shape, temp.dtype)
print("spat :", spat.shape, spat.dtype)
print("y_onehot:", y_onehot.shape, y_onehot.dtype)
print("y_binary:", y_binary.shape, y_binary.dtype)
print("Clases:", np.unique(y_binary, return_counts=True))
print("Sujetos:", len(set(sbjs)))
print("Folds:", len(folds))


# =========================================================
# 8) REPETICIONES
# Cada seed base corre los 5 folds.
# Dentro de cada fold se usa seed + fold.
# =========================================================

all_rows = []
all_run_summaries = []

for rep_id, seed in enumerate(seeds):

    print("\n" + "=" * 90)
    print(f"REPETICIÓN {rep_id + 1}/{len(seeds)} | seed base = {seed}")
    print("=" * 90)

    set_all_seeds(seed)

    run_name = f"{model_name}_{db}_fixed_seed_{seed}"

    results = train_L24O_cv_fixed(
        model_name=model_name,
        X=X,
        y=y_onehot,
        sbjs=sbjs,
        folds=folds,
        model_args=model_args,
        compile_cfg=compile_cfg,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        save_root=SAVE_ROOT,
        run_name=run_name,
        cache_model_format=save_format,
        force_retrain=force_retrain,
        evaluate_test=True,
    )

    # -----------------------------------------------------
    # Tu versión reciente de train_L24O_cv_fixed devuelve:
    # results["fold_results"]
    #
    # Cada fold tiene:
    # fold_result["test_metrics"]
    # -----------------------------------------------------

    fold_results = results["fold_results"]

    for fm in fold_results:
        fold_id = fm["fold"]
        test_metrics = fm["test_metrics"]

        row = {
            "seed": seed,
            "repeat_id": rep_id,
            "fold": fold_id,
            "accuracy": float(test_metrics["accuracy"]),
            "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
            "recall": float(test_metrics["recall"]),
            "precision": float(test_metrics["precision"]),
            "kappa": float(test_metrics["kappa"]),
            "auc": float(test_metrics["auc"]),
        }

        all_rows.append(row)

    run_summary = {
        "seed": seed,
        "repeat_id": rep_id,
        "mean_accuracy_across_5_folds": float(results["mean_accuracy"]),
        "mean_metrics": {
            k: float(v)
            for k, v in results["mean_metrics"].items()
            if isinstance(v, (int, float, np.integer, np.floating))
        },
        "mean_val_metrics": {
            k: float(v)
            for k, v in results["mean_val_metrics"].items()
            if isinstance(v, (int, float, np.integer, np.floating))
        },
        "run_dir": results["run_dir"],
    }

    all_run_summaries.append(run_summary)


# =========================================================
# 9) GUARDAR RESULTADOS CRUDOS
# =========================================================

df = pd.DataFrame(all_rows)
df.to_csv(RESULTS_CSV, index=False)

print("\nArchivo CSV guardado en:")
print(RESULTS_CSV)


# =========================================================
# 10) RESUMEN GLOBAL SOBRE LAS 50 CORRIDAS
# =========================================================

summary_global = {
    "accuracy": summarize_metric(df["accuracy"].tolist()),
    "balanced_accuracy": summarize_metric(df["balanced_accuracy"].tolist()),
    "recall": summarize_metric(df["recall"].tolist()),
    "precision": summarize_metric(df["precision"].tolist()),
    "kappa": summarize_metric(df["kappa"].tolist()),
    "auc": summarize_metric(df["auc"].tolist()),
}


# =========================================================
# 11) RESUMEN POR FOLD
# =========================================================

summary_by_fold = {}

for fold_id in sorted(df["fold"].unique()):
    dff = df[df["fold"] == fold_id]

    summary_by_fold[f"fold_{fold_id}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "balanced_accuracy": summarize_metric(dff["balanced_accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }


# =========================================================
# 12) RESUMEN POR SEED
# =========================================================

summary_by_seed = {}

for seed_val in sorted(df["seed"].unique()):
    dff = df[df["seed"] == seed_val]

    summary_by_seed[f"seed_{seed_val}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "balanced_accuracy": summarize_metric(dff["balanced_accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }


# =========================================================
# 13) GUARDAR JSON FINAL
# =========================================================

final_summary = {
    "model_name": model_name,
    "database": db,
    "model_args": model_args,
    "compile_cfg": compile_cfg_json,

    "input_shapes": {
        "X_raw": list(X_raw.shape),
        "freq": list(freq.shape),
        "temp": list(temp.shape),
        "spat": list(spat.shape),
        "y_onehot": list(y_onehot.shape),
    },

    "n_folds": len(folds),
    "n_repetitions": len(seeds),
    "total_test_runs": int(len(df)),

    "global_summary_over_50_runs": summary_global,
    "summary_by_fold": summary_by_fold,
    "summary_by_seed": summary_by_seed,
    "run_summaries": all_run_summaries,

    "csv_path": RESULTS_CSV,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4)

print("\nArchivo JSON guardado en:")
print(RESULTS_JSON)


# =========================================================
# 14) IMPRIMIR RESULTADOS FINALES
# =========================================================

print("\n" + "=" * 90)
print("RESULTADOS FINALES - MULTI-STREAM TRANSFORMER SIN SCHEDULER")
print("5 FOLDS x 10 REPETICIONES = 50 TESTS")
print("=" * 90)

print(f"Accuracy          : {summary_global['accuracy']['mean']:.4f} ± {summary_global['accuracy']['std_sample']:.4f}")
print(f"Balanced Accuracy : {summary_global['balanced_accuracy']['mean']:.4f} ± {summary_global['balanced_accuracy']['std_sample']:.4f}")
print(f"Recall            : {summary_global['recall']['mean']:.4f} ± {summary_global['recall']['std_sample']:.4f}")
print(f"Precision         : {summary_global['precision']['mean']:.4f} ± {summary_global['precision']['std_sample']:.4f}")
print(f"Kappa             : {summary_global['kappa']['mean']:.4f} ± {summary_global['kappa']['std_sample']:.4f}")
print(f"AUC               : {summary_global['auc']['mean']:.4f} ± {summary_global['auc']['std_sample']:.4f}")

print("\nResumen por fold:")
for fold_name, vals in summary_by_fold.items():
    print(
        f"{fold_name}: "
        f"Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f} | "
        f"Bal Acc = {vals['balanced_accuracy']['mean']:.4f} ± {vals['balanced_accuracy']['std_sample']:.4f}"
    )

print("\nResumen por seed:")
for seed_name, vals in summary_by_seed.items():
    print(
        f"{seed_name}: "
        f"Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f} | "
        f"Bal Acc = {vals['balanced_accuracy']['mean']:.4f} ± {vals['balanced_accuracy']['std_sample']:.4f}"
    )

Using device: cuda
Sujeto eliminado explícitamente: v36p
Sujetos TDAH encontrados: 60
Sujetos Control encontrados: 60
Sujetos totales encontrados: 120
CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - MULTI-STREAM TRANSFORMER
model_args:
  nb_classes: 2
  freq_shape: (20, 1)
  temp_shape: (10, 1)
  spat_shape: (19, 1)
  d_model: 64
  num_layers: 2
  num_heads: 4
  d_ff: 128
  return_dict: True

compile_cfg:
  optimizer_name: adam
  optimizer_params: {'lr': 0.001, 'weight_decay': 0.0}
  scheduler_name: None
  scheduler_params: {}
  early_stopping_patience: 25
  min_delta: 0.0001
  loss_fn: ClassificationLossFromLogits

Datos:
X_raw: (8213, 19, 512) float32
freq : (8213, 20, 1) float32
temp : (8213, 10, 1) float32
spat : (8213, 19, 1) float32
y_onehot: (8213, 2) float32
y_binary: (8213,) int64
Clases: (array([0, 1]), array([3657, 4556]))
Sujetos: 120
Folds: 5

REPETICIÓN 1/10 | seed base = 0

INICIANDO ENTRENAMIENTO CV FIJO - MultiStream
Device: cuda
Run dir: /home/usuario-utp/Desktop/Yessi